# Atelier ML - Analyse d'image : classification

Objectif : entraîner un **réseau de neurones convolutif** (CNN) à reconnaître des vêtements à partir d'images de 28 x 28 pixels, avec le jeu de données **Fashion-MNIST** de Zalando.

## Modules utilisés

- **Pandas** : chargement des fichiers CSV
- **NumPy** : manipulation des tableaux de pixels
- **matplotlib** : affichage des images et des graphiques
- **scikit-learn** : découpage en données d'apprentissage et de validation
- **TensorFlow / Keras** : création et entraînement du réseau de neurones

`tf.config.set_visible_devices([], 'GPU')` force l'utilisation du processeur (CPU) au lieu de la carte graphique.

In [ ]:
import pandas as pnd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
import keras

tf.config.set_visible_devices([], 'GPU')

#Définition de la longueur et de la largeur de l'image
LONGUEUR_IMAGE = 28
LARGEUR_IMAGE = 28

#Chargement des images
observations_entrainement = pnd.read_csv('datas/fashion-mnist_train.csv')

## De quelles données disposons-nous ?

Il n'y a pas de fichiers JPEG ou PNG : chaque image est une **ligne** du fichier CSV.

- 60 000 images d'apprentissage et 10 000 images de test
- chaque image fait 28 x 28 = **784 pixels**
- chaque pixel est une valeur de luminosité entre 0 et 255
- la première colonne (`label`) est le type de vêtement, les 784 autres sont les pixels

| Label | Vêtement |
|---|---|
| 0 | T-shirt/haut |
| 1 | Pantalon |
| 2 | Pull |
| 3 | Robe |
| 4 | Manteau |
| 5 | Sandales |
| 6 | Chemise |
| 7 | Baskets |
| 8 | Sac |
| 9 | Bottes de cheville |

In [ ]:
print(observations_entrainement.shape)
print(observations_entrainement.head())

## Affichage de la première image

On exclut la première colonne (les labels) pour garder seulement les pixels, puis on remet les 784 pixels de la première image en forme de carré 28 x 28 avec `reshape` pour l'afficher.

In [ ]:
#On exclut la première colonne (les labels) pour constituer un tableau de pixels
X = np.array(observations_entrainement.iloc[:, 1:])

premiereImage = X[0]
premiereImage = premiereImage.reshape([LONGUEUR_IMAGE,LARGEUR_IMAGE])
plt.imshow(premiereImage)
plt.show()

La première image est une botte de cheville (label 9). Les données ont été récupérées depuis la source officielle de Keras (`keras.datasets.fashion_mnist`) et converties au format CSV du cours : ce sont les mêmes images, seul l'ordre diffère (dans le cours, la première image est un pull).

## Préparation des données d'apprentissage

`to_categorical` transforme chaque label en un tableau de 10 valeurs avec un 1 à la position de la catégorie :

- label 2 (pull) → `[0,0,1,0,0,0,0,0,0,0]`
- label 9 (bottes) → `[0,0,0,0,0,0,0,0,0,1]`

On découpe ensuite les 60 000 images : **80 %** pour l'apprentissage (48 000) et **20 %** pour la validation (12 000).

In [ ]:
#On crée des catégories à l'aide du module Keras
y = keras.utils.to_categorical(np.array(observations_entrainement.iloc[:,0]))
#Répartition des données d'entrainement en données d'apprentissage et données de validation
#80% de donnée d'apprentissage et 20% de donnée de validation
X_apprentissage, X_validation, y_apprentissage, y_validation = train_test_split(X, y, test_size=0.2, random_state=13)

On remet chaque image au format 28 x 28 x 1 (1 seule couche, car les images sont en niveaux de gris), puis on fait une **mise à l'échelle** (*scaling*) : on divise chaque pixel par 255 pour avoir des valeurs entre 0 et 1. Même traitement pour les données de validation.

In [ ]:
# On redimensionne les images au format 28*28 et on réalise un scaling sur les données des pixels
X_apprentissage = X_apprentissage.reshape(X_apprentissage.shape[0], 
                                          LARGEUR_IMAGE, 
                                          LONGUEUR_IMAGE, 1)
X_apprentissage = X_apprentissage.astype('float32')
X_apprentissage /= 255

X_validation = X_validation.reshape(X_validation.shape[0], 
                                    LARGEUR_IMAGE, 
                                    LONGUEUR_IMAGE, 1)
X_validation = X_validation.astype('float32')
X_validation /= 255

print(X_apprentissage.shape, X_validation.shape)

## Préparation des données de tests

Les 10 000 images de test servent à vérifier l'efficacité du modèle sur des images qu'il n'a jamais vues. Elles sont préparées exactement comme les données d'apprentissage.

In [ ]:
#Preparation des données de tests
observations_test = pnd.read_csv('datas/fashion-mnist_test.csv')
X_test = np.array(observations_test.iloc[:, 1:])
y_test = tf.keras.utils.to_categorical(np.array(observations_test.iloc[:, 0]))
X_test = X_test.reshape(X_test.shape[0], 
                        LARGEUR_IMAGE, 
                        LONGUEUR_IMAGE, 1)
X_test = X_test.astype('float32')
X_test /= 255

print(X_test.shape, y_test.shape)